# TAHAP 3 — Feature Engineering, Encoding & Data Splitting

Membuat fitur perilaku kasir dari data bersih, encode target severity, lalu split data.

**Input:** `data/processed/transactions_cleaned.csv`

**Output:**
- `data/splits/X_train.csv`, `X_val.csv`, `X_test.csv`
- `data/splits/y_train.csv`, `y_val.csv`, `y_test.csv`
- `models/scaler.pkl`
- `models/feature_columns.json`

**9 Fitur Perilaku:**
1. `hour_of_day` — jam transaksi (deteksi aktivitas di luar jam wajar)
2. `is_refund` — flag refund
3. `time_gap_seconds` — jeda antar transaksi per kasir
4. `txn_freq_daily` — frekuensi transaksi harian per kasir
5. `refund_count_daily` — jumlah refund harian per kasir
6. `refund_ratio_daily` — rasio refund harian
7. `amount_zscore_cashier` — z-score nominal relatif per kasir
8. `amount_rolling_mean_5` — rolling mean 5 transaksi terakhir
9. `amount_deviation_from_mean` — deviasi dari rolling mean

## Import & Load Data

In [13]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Paths
INPUT_CSV    = os.path.join("..", "data", "processed", "transactions_cleaned.csv")
SPLITS_DIR   = os.path.join("..", "data", "splits")
MODELS_DIR   = os.path.join("..", "models")
SCALER_PATH  = os.path.join(MODELS_DIR, "scaler.pkl")
FEAT_COL_PATH = os.path.join(MODELS_DIR, "feature_columns.json")

RANDOM_STATE = 42

# Load data bersih
df = pd.read_csv(INPUT_CSV, parse_dates=["timestamp"])
print(f"[OK] Data loaded: {df.shape[0]:,} baris")

[OK] Data loaded: 2,900 baris


## Feature Engineering

In [14]:
print("[1/5] Menghitung fitur perilaku kasir...")

df["date"] = df["timestamp"].dt.date
df = df.sort_values(["cashier_id", "timestamp"])

# 1. Jam transaksi (deteksi aktivitas di luar jam wajar)
df["hour_of_day"] = df["timestamp"].dt.hour

# 2. Penanda refund
df["is_refund"] = (df["transaction_type"] == "REFUND").astype(int)

# 3. Jeda antar transaksi per kasir (deteksi aksi beruntun cepat)
df["time_gap_seconds"] = (
    df.groupby("cashier_id")["timestamp"].diff().dt.total_seconds().fillna(0)
)

# 4. Jumlah transaksi harian per kasir
df["txn_freq_daily"] = (
    df.groupby(["cashier_id", "date"])["id"].transform("count")
)

# 5. Jumlah refund harian per kasir (sinyal kunci untuk fraud halus)
df["refund_count_daily"] = (
    df.groupby(["cashier_id", "date"])["is_refund"].transform("sum")
)

# 6. Rasio refund harian per kasir
df["refund_ratio_daily"] = df["refund_count_daily"] / df["txn_freq_daily"]

# 7. Z-score nominal RELATIF terhadap kebiasaan kasir itu sendiri
g_mean = df.groupby("cashier_id")["amount"].transform("mean")
g_std  = df.groupby("cashier_id")["amount"].transform("std").replace(0, 1)
df["amount_zscore_cashier"] = (df["amount"] - g_mean) / g_std

# 8. Rata-rata bergerak 5 transaksi terakhir per kasir
df["amount_rolling_mean_5"] = (
    df.groupby("cashier_id")["amount"]
      .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

# 9. Deviasi dari rolling mean
df["amount_deviation_from_mean"] = df["amount"] - df["amount_rolling_mean_5"]

# 10. Fitur larut malam
df["is_late_night"] = ((df["hour_of_day"] >= 23) | (df["hour_of_day"] <= 4)).astype(int)

# 11. Z-score dari frekuensi harian per kasir
g_freq_mean = df.groupby("cashier_id")["txn_freq_daily"].transform("mean")
g_freq_std  = df.groupby("cashier_id")["txn_freq_daily"].transform("std").replace(0, 1)
df["txn_freq_zscore_cashier"] = (df["txn_freq_daily"] - g_freq_mean) / g_freq_std

# Daftar kolom fitur
FEATURE_COLS = [
    "hour_of_day",
    "is_refund",
    "time_gap_seconds",
    "txn_freq_daily",
    "refund_count_daily",
    "refund_ratio_daily",
    "amount_zscore_cashier",
    "amount_rolling_mean_5",
    "amount_deviation_from_mean",
    "is_late_night",
    "txn_freq_zscore_cashier",
]

print(f"      {len(FEATURE_COLS)} fitur dihitung")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"      {i:2d}. {col}")

[1/5] Menghitung fitur perilaku kasir...
      11 fitur dihitung
       1. hour_of_day
       2. is_refund
       3. time_gap_seconds
       4. txn_freq_daily
       5. refund_count_daily
       6. refund_ratio_daily
       7. amount_zscore_cashier
       8. amount_rolling_mean_5
       9. amount_deviation_from_mean
      10. is_late_night
      11. txn_freq_zscore_cashier


## Encode Target (Severity → Ordinal)

In [15]:
print("[2/5] Encoding target variable...")

# Mapping severity ke ordinal: LOW=0, MEDIUM=1, HIGH=2, CRITICAL=3
SEVERITY_MAP = {"LOW": 0, "MEDIUM": 1, "HIGH": 2, "CRITICAL": 3}
SEVERITY_LABELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

df["severity_encoded"] = df["fraud_severity"].map(SEVERITY_MAP)

print(f"      Mapping: {SEVERITY_MAP}")
print(f"\n      Distribusi target:")
for label, code in SEVERITY_MAP.items():
    cnt = (df["severity_encoded"] == code).sum()
    print(f"        {label} ({code}): {cnt:,}")

[2/5] Encoding target variable...
      Mapping: {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2, 'CRITICAL': 3}

      Distribusi target:
        LOW (0): 2,500
        MEDIUM (1): 180
        HIGH (2): 120
        CRITICAL (3): 100


## Split Data (Stratified 70/15/15)

In [16]:
print("[3/5] Membagi data: Train 70% / Validation 15% / Test 15%")

# Siapkan X dan y
X = df[FEATURE_COLS].fillna(0)
y = df["severity_encoded"]

# Split pertama: train (70%) vs temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Split kedua: temp → validation (15%) + test (15%)
# 50% dari 30% = 15% masing-masing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"      Train     : {len(X_train):,} baris")
print(f"      Validation: {len(X_val):,} baris")
print(f"      Test      : {len(X_test):,} baris")

# Cek proporsi severity di setiap split
print(f"\n      Proporsi severity per split:")
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = y_split.value_counts().sort_index()
    parts = [f"{SEVERITY_LABELS[i]}={c}" for i, c in counts.items()]
    print(f"        {name:5s}: {', '.join(parts)}")

[3/5] Membagi data: Train 70% / Validation 15% / Test 15%
      Train     : 2,030 baris
      Validation: 435 baris
      Test      : 435 baris

      Proporsi severity per split:
        Train: LOW=1750, MEDIUM=126, HIGH=84, CRITICAL=70
        Val  : LOW=375, MEDIUM=27, HIGH=18, CRITICAL=15
        Test : LOW=375, MEDIUM=27, HIGH=18, CRITICAL=15


## Scaling (RobustScaler)

In [17]:
print("[4/5] Scaling fitur dengan RobustScaler...")

# Fit scaler HANYA pada training data (mencegah data leakage)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=FEATURE_COLS,
    index=X_train.index
)

# Transform validation dan test dengan scaler yang sama
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=FEATURE_COLS,
    index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=FEATURE_COLS,
    index=X_test.index
)

print(f"      Scaler fitted pada {len(X_train):,} baris training data")
print(f"      Median (train): {scaler.center_[:3]}...")
print(f"      Scale  (train): {scaler.scale_[:3]}...")

[4/5] Scaling fitur dengan RobustScaler...
      Scaler fitted pada 2,030 baris training data
      Median (train): [  17.     0.  1535.5]...
      Scale  (train): [1.70000e+01 1.00000e+00 3.29975e+03]...


## Simpan Splits & Artifacts

In [18]:
print("[5/5] Menyimpan splits & artifacts...")

# Buat direktori
os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Simpan splits (scaled) ke CSV
X_train_scaled.to_csv(os.path.join(SPLITS_DIR, "X_train.csv"), index=False)
X_val_scaled.to_csv(os.path.join(SPLITS_DIR, "X_val.csv"), index=False)
X_test_scaled.to_csv(os.path.join(SPLITS_DIR, "X_test.csv"), index=False)

y_train.to_csv(os.path.join(SPLITS_DIR, "y_train.csv"), index=False, header=["severity_encoded"])
y_val.to_csv(os.path.join(SPLITS_DIR, "y_val.csv"), index=False, header=["severity_encoded"])
y_test.to_csv(os.path.join(SPLITS_DIR, "y_test.csv"), index=False, header=["severity_encoded"])

print(f"      Splits disimpan ke {SPLITS_DIR}")

# Simpan scaler
joblib.dump(scaler, SCALER_PATH)
print(f"      Scaler disimpan: {SCALER_PATH}")

# Simpan daftar fitur + metadata ke JSON
feature_meta = {
    "feature_columns": FEATURE_COLS,
    "severity_map": SEVERITY_MAP,
    "severity_labels": SEVERITY_LABELS,
    "n_features": len(FEATURE_COLS),
}
with open(FEAT_COL_PATH, "w") as f:
    json.dump(feature_meta, f, indent=2)
print(f"      Feature columns disimpan: {FEAT_COL_PATH}")

print("\nSemua artifacts siap untuk Tahap 4 (Model Training)")

[5/5] Menyimpan splits & artifacts...
      Splits disimpan ke ..\data\splits
      Scaler disimpan: ..\models\scaler.pkl
      Feature columns disimpan: ..\models\feature_columns.json

Semua artifacts siap untuk Tahap 4 (Model Training)
